# LeetCode 2437 — Number of Valid Clock Times bằng Python

Link đề: https://leetcode.com/problems/number-of-valid-clock-times/ · Độ khó: **Easy** · Chủ đề: String, Enumeration, Counting

Skeleton LeetCode:

```python
class Solution:
    def countTime(self, time: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán: chuỗi `"hh:mm"` có một số chữ số bị che bằng `?`; đếm có bao nhiêu **giờ hợp lệ** (`00:00` đến `23:59`) khi thay `?` bằng chữ số.
2. Dùng **quy tắc nhân (counting)**: số cách chọn giờ **độc lập** với số cách chọn phút nên đáp án = `đếm_giờ × đếm_phút`.
3. Phân tích từng chữ số: chữ số giờ đầu, chữ số giờ thứ hai (phụ thuộc chữ số đầu), phút đầu (`0..5`), phút thứ hai (`0..9`).
4. Viết 4 cách: duyệt cả 1440 phút trong ngày, sinh tổ hợp `itertools.product`, đếm giờ và phút riêng, và công thức theo luật chữ số; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: nhân 10 cho mọi `?`, quên điều kiện của chữ số giờ khi chữ số kia cố định, đặt sai cận chữ số phút đầu.

## 2. Đề bài & ví dụ

Cho chuỗi `time` có dạng `"hh:mm"`. Một số chữ số có thể bị thay bằng `?`. Các chữ số `?` có thể thay bằng bất kỳ chữ số từ `0` đến `9`.

Giờ hợp lệ nằm trong khoảng từ `"00:00"` đến `"23:59"`. Hãy trả về **số lượng thời gian hợp lệ khác nhau** có thể tạo ra bằng cách thay mọi `?`.

| Ví dụ | time | Output | Giải thích |
|---|---|---|---|
| 1 | `"?5:00"` | 2 | `"05:00"` và `"15:00"` hợp lệ; `"25:00"` thì không |
| 2 | `"0?:0?"` | 100 | giờ `00..09` (10 cách) × phút `00..09` (10 cách) |
| 3 | `"??:??"` | 1440 | cả ngày `24 × 60` |

Constraints: `time` có dạng `"hh:mm"`; bảo đảm **ít nhất một** thời gian hợp lệ có thể tạo ra.

In [1]:
for t in ["?5:00", "0?:0?", "??:??"]:
    print(repr(t), "len =", len(t), "| các chỉ số chữ số: 0,1 (giờ) và 3,4 (phút) | dấu ':' ở chỉ số 2:", t[2])

'?5:00' len = 5 | các chỉ số chữ số: 0,1 (giờ) và 3,4 (phút) | dấu ':' ở chỉ số 2: :
'0?:0?' len = 5 | các chỉ số chữ số: 0,1 (giờ) và 3,4 (phút) | dấu ':' ở chỉ số 2: :
'??:??' len = 5 | các chỉ số chữ số: 0,1 (giờ) và 3,4 (phút) | dấu ':' ở chỉ số 2: :


## 3. Kiến thức nền

### 3.1. Chỉ số trong chuỗi `"hh:mm"`

`time[0]`, `time[1]` là hai chữ số giờ, `time[2]` là `':'`, `time[3]`, `time[4]` là hai chữ số phút. Lát cắt: `time[:2]` là giờ, `time[3:]` là phút.

In [2]:
t = "?5:00"
print(t[0], t[1], t[2], t[3], t[4], "| giờ:", t[:2], "| phút:", t[3:])

? 5 : 0 0 | giờ: ?5 | phút: 00


### 3.2. Quy tắc nhân (counting)

Nếu có `A` cách chọn việc thứ nhất và `B` cách chọn việc thứ hai, **độc lập** nhau, tổng cách chọn là `A × B`. Giờ và phút độc lập (mọi giờ đều ghép được mọi phút), nên `đáp án = (số giờ) × (số phút)`.

In [3]:
gio, phut = 2, 60
print("số cách ghép giờ x phút:", gio * phut)
print("kiểm tra bằng đếm cặp:", sum(1 for g in range(gio) for p in range(phut)))

số cách ghép giờ x phút: 120
kiểm tra bằng đếm cặp: 120


### 3.3. So khớp mẫu có `?` bằng so sánh từng ký tự

Một chuỗi `candidate` khớp mẫu `pattern` nếu **mỗi vị trí**: `pattern[i] == '?'` hoặc `pattern[i] == candidate[i]`. Dùng `zip(pattern, candidate)` và `all(...)`.

In [4]:
def matches(pattern, candidate):
    return all(p == '?' or p == c for p, c in zip(pattern, candidate))

print(matches("?5:00", "05:00"), matches("?5:00", "15:00"), matches("?5:00", "05:01"))

True True False


### 3.4. Định dạng số có đệm 0: `f"{h:02d}"`

`f"{7:02d}"` cho `"07"`: rộng tối thiểu 2 ký tự, đệm `0` bên trái. Hữu ích để tạo chuỗi `"hh:mm"` từ hai số nguyên.

In [5]:
print(f"{7:02d}:{5:02d}", f"{23:02d}:{59:02d}", f"{0:02d}:{0:02d}")

07:05 23:59 00:00


### 3.5. `itertools.product`

`product(range(10), repeat=2)` sinh mọi cặp chữ số `(0,0), (0,1), ..., (9,9)` (100 cặp). Với `k` ký tự `?` ta có `10^k` tổ hợp (tối đa `10^4` ở bài này) — đủ nhỏ để thử hết.

In [6]:
from itertools import product
print(len(list(product(range(10), repeat=2))), list(product("01", repeat=2)))

100 [('0', '0'), ('0', '1'), ('1', '0'), ('1', '1')]


### 3.6. Điều kiện hợp lệ của từng chữ số

- Chữ số giờ thứ nhất `h0` thuộc `{0, 1, 2}`.
- Nếu `h0 ∈ {0, 1}` thì `h1` tuỳ ý `0..9`; nếu `h0 = 2` thì `h1 ∈ {0, 1, 2, 3}`.
- Chữ số phút đầu `m0 ∈ {0, ..., 5}` (phút `<= 59`); `m1` tuỳ ý `0..9`.

Nghĩa là hai chữ số giờ **phụ thuộc nhau**, còn hai chữ số phút thì **độc lập**.

In [7]:
valid_hours = [h for h in range(24)]
print("h0 có thể:", sorted({h // 10 for h in valid_hours}))
print("h1 khi h0=2:", [h % 10 for h in valid_hours if h // 10 == 2])
print("m0 có thể:", sorted({m // 10 for m in range(60)}))

h0 có thể: [0, 1, 2]
h1 khi h0=2: [0, 1, 2, 3]
m0 có thể: [0, 1, 2, 3, 4, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chỉ có `24 × 60 = 1440` thời gian hợp lệ trong một ngày. Ta có thể **liệt kê tất cả** rồi đếm cái nào khớp mẫu có `?`.

Ví dụ `"?5:00"`:

```
05:00 ✓ (0 thay ?)   15:00 ✓ (1 thay ?)   25:00 ✗ (không là giờ hợp lệ)
```

Đếm = 2. Nhưng cũng có thể suy luận trực tiếp: giờ có dạng `?5` — thử `h0 = 0, 1, 2`: `05`, `15`, `25`; chỉ `05` và `15` hợp lệ (`25 > 23`). Phút `00` cố định → 1 cách. Vậy `2 × 1 = 2`.

Ví dụ `"0?:0?"`: giờ `0?` có 10 cách (`00..09` đều hợp lệ), phút `0?` có 10 cách (`00..09`) → `10 × 10 = 100`.

Nhận xét: nên **tách giờ và phút**, đếm riêng rồi nhân.

## 5. Cách 1 — Brute Force / baseline

Duyệt tất cả `1440` thời gian hợp lệ `hh:mm` (hai vòng lặp `h` trong `0..23`, `m` trong `0..59`), tạo chuỗi và kiểm tra có khớp mẫu không. Đếm.

Đúng đề tuyệt đối, khó sai. Nút thắt: luôn làm 1440 lần dù mẫu chỉ có vài dấu `?`.

In [8]:
class SolutionBrute:
    def countTime(self, time: str) -> int:
        count = 0
        for h in range(24):
            for m in range(60):
                cand = f"{h:02d}:{m:02d}"
                ok = True
                for p, c in zip(time, cand):
                    if p != '?' and p != c:
                        ok = False
                        break
                if ok:
                    count += 1
        return count

print(SolutionBrute().countTime("?5:00"))
print(SolutionBrute().countTime("0?:0?"))
print(SolutionBrute().countTime("??:??"))

2
100
1440


**Độ phức tạp:** `1440` thời gian × `5` ký tự so khớp → `O(1440 · 5)`, tức hằng số `O(1)`. Space `O(1)`. Nút thắt: không tận dụng cấu trúc "giờ × phút độc lập".

## 6. Ý tưởng thuật toán / bước đột phá

Giờ (hai chữ số đầu) và phút (hai chữ số cuối) **độc lập**: `đáp án = đếm_giờ × đếm_phút`. Rồi từng phần có quy tắc theo chữ số:

**Đếm giờ** (`h0 = time[0]`, `h1 = time[1]`):

- Cả hai là `?` → `24` (giờ `00..23`).
- Chỉ `h0 = ?`: nếu `h1 <= '3'` thì `h0` có thể `0, 1, 2` → `3` cách; nếu `h1 >= '4'` thì `h0` chỉ `0, 1` → `2` cách.
- Chỉ `h1 = ?`: nếu `h0 == '2'` thì `h1` thuộc `0..3` → `4` cách; ngược lại (`h0 ∈ {0, 1}`) → `10` cách.
- Cả hai cố định → `1` (đề bảo đảm hợp lệ).

**Đếm phút** (`m0 = time[3]`, `m1 = time[4]`):

- `m0 = ?` → `6` cách (`0..5`), ngược lại `1`.
- `m1 = ?` → `10` cách, ngược lại `1`.
- Hai phần độc lập nên nhân.

Invariant: mỗi nhóm (giờ, phút) được đếm đúng số giá trị hợp lệ thoả mẫu; và mọi sự kết hợp giờ-phút là hợp lệ.

In [9]:
def count_hours(h0, h1):
    if h0 == '?' and h1 == '?':
        return 24
    if h0 == '?':
        return 3 if h1 <= '3' else 2
    if h1 == '?':
        return 4 if h0 == '2' else 10
    return 1

def count_minutes(m0, m1):
    return (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)

for t in ["?5:00", "0?:0?", "??:??", "2?:?5", "?3:5?"]:
    print(t, "->", count_hours(t[0], t[1]), "x", count_minutes(t[3], t[4]), "=", count_hours(t[0], t[1]) * count_minutes(t[3], t[4]))

?5:00 -> 2 x 1 = 2
0?:0? -> 10 x 10 = 100
??:?? -> 24 x 60 = 1440
2?:?5 -> 4 x 6 = 24
?3:5? -> 3 x 10 = 30


## 7. Các cách tiếp theo

Ta có 4 cách: (1) duyệt cả ngày, (2) `product` thay `?`, (3) đếm giờ và phút riêng bằng liệt kê, (4) công thức theo luật chữ số.

### Cách 2 — Thay `?` bằng `product` rồi kiểm tra hợp lệ

Tìm các vị trí `?`, với mỗi tổ hợp chữ số điền vào, dựng chuỗi mới và kiểm tra `hh <= 23` và `mm <= 59`. Tối đa `10^4` tổ hợp.

In [10]:
from itertools import product

class SolutionProduct:
    def countTime(self, time: str) -> int:
        qs = [i for i, ch in enumerate(time) if ch == '?']
        count = 0
        for digits in product("0123456789", repeat=len(qs)):
            chars = list(time)
            for pos, d in zip(qs, digits):
                chars[pos] = d
            hh, mm = int("".join(chars[:2])), int("".join(chars[3:]))
            if hh <= 23 and mm <= 59:
                count += 1
        return count

print(SolutionProduct().countTime("?5:00"), SolutionProduct().countTime("0?:0?"), SolutionProduct().countTime("??:??"))

2 100 1440


Độ phức tạp Cách 2: Time `O(10^k · 5)` với `k` là số dấu `?` (`k <= 4`, tối đa `10^4`), Space `O(k)`.

### Cách 3 — Đếm giờ và phút riêng bằng liệt kê

Đếm số `h` trong `0..23` khớp `time[:2]` và số `m` trong `0..59` khớp `time[3:]`, rồi nhân. Tận dụng tính độc lập, nhưng vẫn liệt kê thay vì suy luận.

In [11]:
class SolutionSplit:
    def countTime(self, time: str) -> int:
        def fit(pat, cand):
            return all(p == '?' or p == c for p, c in zip(pat, cand))
        hours = sum(1 for h in range(24) if fit(time[:2], f"{h:02d}"))
        minutes = sum(1 for m in range(60) if fit(time[3:], f"{m:02d}"))
        return hours * minutes

print(SolutionSplit().countTime("?5:00"), SolutionSplit().countTime("0?:0?"), SolutionSplit().countTime("??:??"))

2 100 1440


Độ phức tạp Cách 3: Time `O(24 + 60) = O(1)`, Space `O(1)`.

### Cách 4 — Công thức theo luật chữ số (cách chuẩn)

Dùng đúng các quy tắc "nếu… thì…" ở mục 6. Không vòng lặp, `O(1)` rõ ràng.

In [12]:
class SolutionRule:
    def countTime(self, time: str) -> int:
        h0, h1, m0, m1 = time[0], time[1], time[3], time[4]
        if h0 == '?' and h1 == '?':
            hours = 24
        elif h0 == '?':
            hours = 3 if h1 <= '3' else 2
        elif h1 == '?':
            hours = 4 if h0 == '2' else 10
        else:
            hours = 1
        minutes = (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)
        return hours * minutes

print(SolutionRule().countTime("?5:00"), SolutionRule().countTime("0?:0?"), SolutionRule().countTime("??:??"))

2 100 1440


Độ phức tạp Cách 4: Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 với `time = "2?:?5"`.

| bước | giá trị |
|---|---|
| `h0, h1, m0, m1` | `'2', '?', '?', '5'` |
| nhánh giờ | `h1 == '?'` và `h0 == '2'` → `hours = 4` (`20, 21, 22, 23`) |
| `m0 == '?'` | `6` cách (`0..5`) |
| `m1 == '5'` | `1` |
| `minutes` | `6 × 1 = 6` |
| đáp án | `4 × 6 = 24` |

In [13]:
def trace_rule(time):
    h0, h1, m0, m1 = time[0], time[1], time[3], time[4]
    print(f"time={time!r}: h0={h0!r} h1={h1!r} m0={m0!r} m1={m1!r}")
    if h0 == '?' and h1 == '?':
        hours, why = 24, "cả hai ?"
    elif h0 == '?':
        hours, why = (3, "h1 <= 3 -> h0 in 0,1,2") if h1 <= '3' else (2, "h1 >= 4 -> h0 in 0,1")
    elif h1 == '?':
        hours, why = (4, "h0 == 2 -> h1 in 0..3") if h0 == '2' else (10, "h0 in 0,1 -> h1 in 0..9")
    else:
        hours, why = 1, "cố định"
    print(f"  giờ  : {hours} ({why})")
    minutes = (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)
    print(f"  phút : {minutes}")
    print(f"  đáp án = {hours} x {minutes} = {hours * minutes}")
    return hours * minutes

trace_rule("2?:?5")
trace_rule("?5:00")
trace_rule("?4:?0")

time='2?:?5': h0='2' h1='?' m0='?' m1='5'
  giờ  : 4 (h0 == 2 -> h1 in 0..3)
  phút : 6
  đáp án = 4 x 6 = 24
time='?5:00': h0='?' h1='5' m0='0' m1='0'
  giờ  : 2 (h1 >= 4 -> h0 in 0,1)
  phút : 1
  đáp án = 2 x 1 = 2
time='?4:?0': h0='?' h1='4' m0='?' m1='0'
  giờ  : 2 (h1 >= 4 -> h0 in 0,1)
  phút : 6
  đáp án = 2 x 6 = 12


12

## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def countTime(self, time: str) -> int:
        h0, h1, m0, m1 = time[0], time[1], time[3], time[4]
        if h0 == '?' and h1 == '?':
            hours = 24
        elif h0 == '?':
            hours = 3 if h1 <= '3' else 2
        elif h1 == '?':
            hours = 4 if h0 == '2' else 10
        else:
            hours = 1
        minutes = (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)
        return hours * minutes

print(Solution().countTime("?5:00"), Solution().countTime("0?:0?"), Solution().countTime("??:??"))

2 100 1440


## 10. Vì sao thuật toán đúng

- **Độc lập giờ-phút:** điều kiện hợp lệ của giờ chỉ liên quan `h0, h1` và điều kiện của phút chỉ liên quan `m0, m1`. Mọi cặp (giờ hợp lệ, phút hợp lệ) là một thời gian hợp lệ → quy tắc nhân đúng.
- **Phút:** `m0 ∈ 0..5` (6 giá trị) và `m1 ∈ 0..9` (10 giá trị) cho đủ `60` phút `00..59`; mỗi chữ số chỉ phụ thuộc chính nó nên nhân nhau.
- **Giờ, `h0 = ?` và `h1` đã biết:** thử `h0 ∈ {0,1,2}`: `h0 ∈ {0,1}` luôn hợp lệ với mọi `h1 ∈ 0..9`; `h0 = 2` chỉ hợp lệ khi `h1 ≤ 3`. Vậy có `3` cách nếu `h1 ≤ 3`, ngược lại `2`.
- **Giờ, `h1 = ?` và `h0` đã biết:** `h0 ∈ {0,1}` → `h1` tuỳ ý (10); `h0 = 2` → `h1 ∈ 0..3` (4).
- **Cả hai `?`:** đúng `24` giờ. **Cả hai cố định:** `1` (đề bảo đảm hợp lệ).
- **Phủ hết trường hợp:** bốn nhánh giờ là phân hoạch đầy đủ của `(h0 = ?, h1 = ?)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt cả ngày | `O(1440 · 5)` (hằng số) | `O(1)` |
| 2. `product` thay `?` | `O(10^k · 5)`, `k <= 4` | `O(k)` |
| 3. Đếm giờ/phút riêng | `O(24 + 60)` | `O(1)` |
| 4. Công thức luật chữ số | `O(1)` | `O(1)` |

Độ dài chuỗi cố định (5 ký tự), mọi cách đều là hằng số. Sự khác nhau là **hằng số** và độ dễ sai. Không có khác biệt average/worst.

## 12. Bộ test

Test 1: ví dụ đề, biên và các dạng `?` khác nhau.

In [15]:
impls = [SolutionBrute(), SolutionProduct(), SolutionSplit(), SolutionRule(), Solution()]
tests = [
    ("?5:00", 2),
    ("0?:0?", 100),
    ("??:??", 1440),
    ("00:00", 1),          # không có ?
    ("23:59", 1),
    ("2?:00", 4),          # 20..23 (bắt lỗi h1 luôn 10)
    ("?3:00", 3),          # 03, 13, 23
    ("?4:00", 2),          # 04, 14 (24 không hợp lệ)
    ("1?:5?", 100),
    ("??:5?", 240),
    ("2?:??", 240),        # 4 giờ x 60 phút
    ("?9:?9", 12),         # 2 giờ x 6 phút
    ("0?:?0", 60),         # 10 giờ x 6 phút
    ("?0:0?", 30),         # 3 giờ x 10 phút
]
for t, exp in tests:
    for im in impls:
        assert im.countTime(t) == exp, (type(im).__name__, t, exp, im.countTime(t))
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 14


Test 2: vét cạn — mọi mẫu `hh:mm` có thể tạo bằng cách che ngẫu nhiên các chữ số của **mọi** giờ hợp lệ (đảm bảo ít nhất một thời gian hợp lệ như đề). Ngẫu nhiên có seed, đối chiếu với oracle "duyệt cả ngày".

In [16]:
import random
random.seed(2437)

def oracle(pattern):
    cnt = 0
    for h in range(24):
        for m in range(60):
            cand = f"{h:02d}:{m:02d}"
            if all(p == '?' or p == c for p, c in zip(pattern, cand)):
                cnt += 1
    return cnt

seen = set()
for _ in range(2500):
    h, m = random.randint(0, 23), random.randint(0, 59)
    chars = list(f"{h:02d}:{m:02d}")
    prob = random.choice([0.2, 0.5, 0.8])
    for i in (0, 1, 3, 4):
        if random.random() < prob:
            chars[i] = '?'
    pat = ''.join(chars)
    assert len(pat) == 5 and pat[2] == ':'
    exp = oracle(pat)
    assert exp >= 1                      # bảo đảm của đề
    seen.add(pat)
    for im in impls:
        assert im.countTime(pat) == exp, (type(im).__name__, pat, exp)
print("2500 test ngẫu nhiên OK, số mẫu khác nhau:", len(seen))

2500 test ngẫu nhiên OK, số mẫu khác nhau: 1013


Test 3: kiểm tra có hệ thống mọi mẫu ghép từ tập chữ số giờ `"012?"` / `"034?9"` và chữ số phút `"05?"` / `"09?"`, bỏ qua mẫu không thể tạo giờ hợp lệ (vì đề bảo đảm có ít nhất một thời gian hợp lệ).

In [17]:
count_checked = 0
for h0 in "012?":
    for h1 in "034?9":
        for m0 in "05?":
            for m1 in "09?":
                pat = f"{h0}{h1}:{m0}{m1}"
                exp = oracle(pat)
                if exp == 0:
                    continue                         # đề đảm bảo có ít nhất một thời gian hợp lệ
                count_checked += 1
                assert Solution().countTime(pat) == exp, pat
print("đã kiểm tra", count_checked, "mẫu có đáp án >= 1: OK")

đã kiểm tra 162 mẫu có đáp án >= 1: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhân 10 cho mỗi dấu `?`

In [18]:
def wrong_ten(time):
    return 10 ** time.count('?')               # SAI: bỏ qua giới hạn của giờ và phút

print("sai :", wrong_ten("?5:00"), wrong_ten("??:??"))
print("đúng:", Solution().countTime("?5:00"), Solution().countTime("??:??"))
assert wrong_ten("?5:00") != Solution().countTime("?5:00")
assert wrong_ten("??:??") != Solution().countTime("??:??")

sai : 10 10000
đúng: 2 1440


Vì sao sai: không phải chữ số nào cũng được `0..9`; giờ tối đa `23`, phút tối đa `59`. `"??:??"` chỉ có `1440` thời gian chứ không phải `10000`. **Sửa:** áp dụng giới hạn theo từng vị trí.

### Lỗi 2: `h0 = ?` luôn cho `3` cách, không xét `h1`

In [19]:
def wrong_h0(time):
    h0, h1, m0, m1 = time[0], time[1], time[3], time[4]
    hours = 3 if h0 == '?' and h1 != '?' else (24 if h0 == '?' else (4 if h1 == '?' and h0 == '2' else (10 if h1 == '?' else 1)))
    minutes = (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)
    return hours * minutes

print("sai :", wrong_h0("?5:00"))
print("đúng:", Solution().countTime("?5:00"))
assert wrong_h0("?5:00") != Solution().countTime("?5:00")

sai : 3
đúng: 2


Vì sao sai: nếu `h1 = 5` thì `h0 = 2` cho `25` không hợp lệ, nên chỉ có `h0 ∈ {0, 1}` → `2` cách. Nghĩa là số cách của `h0` **phụ thuộc `h1`** (ngưỡng `3`). **Sửa:** `3 if h1 <= '3' else 2`.

### Lỗi 3: `h1 = ?` luôn cho `10` cách, không xét `h0 == '2'`

In [20]:
def wrong_h1(time):
    h0, h1, m0, m1 = time[0], time[1], time[3], time[4]
    if h0 == '?' and h1 == '?':
        hours = 24
    elif h0 == '?':
        hours = 3 if h1 <= '3' else 2
    elif h1 == '?':
        hours = 10                             # SAI: khi h0 = '2' chỉ còn 4 cách
    else:
        hours = 1
    return hours * (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)

print("sai :", wrong_h1("2?:00"))
print("đúng:", Solution().countTime("2?:00"))
assert wrong_h1("2?:00") != Solution().countTime("2?:00")

sai : 10
đúng: 4


Vì sao sai: `"2?:00"` chỉ có `20, 21, 22, 23` (4 giờ), vì `24..29` không tồn tại. **Sửa:** `4 if h0 == '2' else 10`.

### Lỗi 4 (bonus): cận chữ số phút đầu

Chữ số phút đầu chỉ `0..5` (6 giá trị), **không** phải 10. Viết `10` cho `m0 == '?'` sẽ ra `"00:?0"` → `10` thay vì `6`. Hãy nhớ: phút lớn nhất là `59`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu cố định nhỏ (một ngày chỉ có `1440` phút) → **vét cạn có kiểm tra khớp mẫu** luôn là phương án dự phòng an toàn.
- Hai phần độc lập → dùng **quy tắc nhân**: `đáp án = đếm_phần_1 × đếm_phần_2`.
- Với chữ số có ràng buộc "phụ thuộc" (như `h0` và `h1`), liệt kê các trường hợp `?` và chữ số cố định rồi tự đếm bằng tay.
- Dùng `f"{h:02d}"` để tạo `"hh"` đệm số 0 mà không phải tự ghép chuỗi.
- Khi trực giác không chắc, hãy so sánh công thức với oracle vét cạn ở mọi mẫu nhỏ (như Test 3).
- So sánh ký tự số bằng chuỗi (`h1 <= '3'`) hợp lệ vì `'0' < '1' < ... < '9'` theo bảng mã.

## 15. Biến thể của bài

1. Đồng hồ 12 giờ (`01..12`) với `AM/PM`.
2. Cho đồng hồ **có giây** `"hh:mm:ss"` (thêm một thành phần `00..59`).
3. Liệt kê (không chỉ đếm) mọi thời gian hợp lệ theo thứ tự tăng dần (hoặc tìm **thời gian lớn nhất** — bài "Latest Time by Replacing Hidden Digits").
4. Tìm thời gian **sớm nhất / muộn nhất** từ 4 chữ số cho trước (bài "Largest Time for Given Digits").
5. Đếm ngày hợp lệ `DD/MM` từ mẫu có `?`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt cả ngày | thử 1440 thời gian | hằng số lớn | `O(1)` | oracle test, không muốn suy luận |
| 2. `product` thay `?` | sinh tổ hợp chữ số, kiểm tra | `O(10^k)` | `O(k)` | tổng quát khi ràng buộc phức tạp |
| 3. Đếm giờ/phút riêng | liệt kê 24 + 60 rồi nhân | `O(84)` | `O(1)` | cân bằng dễ viết và nhanh |
| 4. Công thức luật chữ số | `if/elif` theo vị trí `?` | `O(1)` | `O(1)` | **chuẩn**, nhanh và gọn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `countTime("1?:?3")` bằng bao nhiêu?

**Bài 2 (tính tay):** `"?9:5?"` có bao nhiêu thời gian hợp lệ? Viết ra các giờ và các phút.

**Bài 3 (điền TODO):** hoàn thành `count_minutes_only(m0, m1)` trả về số cách của phần phút theo mẫu.

**Bài 4 (tự viết lại):** viết `all_times(time)` trả về **danh sách** các thời gian hợp lệ (chuỗi `"hh:mm"`) theo thứ tự tăng dần; kiểm tra `len(all_times(t)) == Solution().countTime(t)`.

**Bài 5 (mini challenge):** viết `latest_time(time)` trả về thời gian **lớn nhất** có thể (thay mọi `?`).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().countTime("1?:?3"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_minutes_only(m0, m1):
    # TODO: m0 == '?' -> 6 cách, m1 == '?' -> 10 cách; còn lại 1; nhân hai phần
    pass

60


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** tách giờ và phút, đếm riêng rồi nhân.

**Mức 2 (vừa):** Bài 3: `(6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)`. Bài 4: duyệt `h` trong `0..23`, `m` trong `0..59`, lọc chuỗi khớp mẫu. Bài 5: `max` của `all_times(time)` (hoặc chọn chữ số lớn nhất ở từng vị trí).

**Mức 3 (gần đáp án):** Bài 5 đơn giản nhất: `return max(all_times(time))` (chuỗi `hh:mm` đệm số 0 nên so sánh chuỗi cũng đúng thứ tự thời gian). Bài 2: giờ `?9` → `09`, `19` (2 giờ); phút `5?` → `50..59` (10 phút) → `20`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: giờ 1? -> 10; phút ?3 -> 6 -> 60
assert Solution().countTime("1?:?3") == 60

# Bài 2: giờ 09, 19; phút 50..59 -> 20
assert Solution().countTime("?9:5?") == 20

# Bài 3
def count_minutes_only(m0, m1):
    return (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)
assert count_minutes_only('?', '?') == 60
assert count_minutes_only('3', '?') == 10
assert count_minutes_only('?', '7') == 6

# Bài 4
def all_times(time):
    res = []
    for h in range(24):
        for m in range(60):
            cand = f"{h:02d}:{m:02d}"
            if all(p == '?' or p == c for p, c in zip(time, cand)):
                res.append(cand)
    return res
assert all_times("?5:0?")[:3] == ["05:00", "05:01", "05:02"]
for t in ["?5:00", "2?:?5", "??:??", "1?:3?"]:
    assert len(all_times(t)) == Solution().countTime(t)

# Bài 5
def latest_time(time):
    return max(all_times(time))
assert latest_time("??:??") == "23:59"
assert latest_time("?5:?0") == "15:50"
assert latest_time("0?:?4") == "09:54"
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm theo luật chữ số":**

```python
h0, h1, m0, m1 = time[0], time[1], time[3], time[4]
if h0 == '?' and h1 == '?': hours = 24
elif h0 == '?':             hours = 3 if h1 <= '3' else 2
elif h1 == '?':             hours = 4 if h0 == '2' else 10
else:                       hours = 1
minutes = (6 if m0 == '?' else 1) * (10 if m1 == '?' else 1)
return hours * minutes
```

**Checklist trước khi submit:**

- [ ] Đã tách giờ và phút rồi **nhân** (không cộng)?
- [ ] Chữ số phút đầu `?` có `6` cách (không phải 10)?
- [ ] `h0 = ?` xét theo `h1 <= '3'`; `h1 = ?` xét theo `h0 == '2'`?
- [ ] Trường hợp `"??:??"` ra `1440`?
- [ ] Đã so sánh với oracle vét cạn trên nhiều mẫu ngẫu nhiên có seed?
- [ ] Chữ ký đúng `countTime(self, time: str) -> int`?